<!-- circuitkit-logo -->
<p align="center">
  <picture>
    <source media="(prefers-color-scheme: dark)" srcset="../../docs/assets/circuitkit-logo-white.png">
    <img src="../../docs/assets/circuitkit-logo-black.png" width="220" alt="CircuitKIT">
  </picture>
</p>

# 🔬 Advanced Research Tools

This notebook covers research-grade tooling for paper reproducibility and deeper analysis — not part of the everyday Discover → Evaluate → Intervene loop.

- **Section A:** Selector Registry — all 14 selectors, custom registration
- **Section B:** MasterGrid — (methods × wrappers × seeds) experiment grid
- **Section C:** Intervention Faithfulness (IF) — LOO cross-validated metric
- **Section D:** Pipeline reuse — `from_artifact` / `from_scores` iteration

- **Model:** GPT-2 (CPU-friendly where possible)
- **Runtime:** ~10 min (Sections B/C use small grids or synthetic data)

---

## Setup

In [1]:
import warnings
warnings.filterwarnings('ignore')

import circuitkit as ck
from circuitkit import Pipeline

---
## Section A: Selector Registry

CircuitKIT has 14 registered selectors — functions that rank nodes by different criteria (magnitude, random, algorithm-specific, etc.). They're used internally by Pillar 5 (baselines) and can be used directly for custom experiments.

In [2]:
from circuitkit.selection import list_selectors, get_selector

# List all registered selectors
all_selectors = list_selectors()
print(f"{len(all_selectors)} selectors registered:")
for name in sorted(all_selectors):
    print(f"  - {name}")

14 selectors registered:
  - awq
  - cdt
  - eap
  - eap-gp
  - eap-ig
  - gptq
  - ibcircuit
  - magnitude
  - multi_granular
  - random
  - relp
  - tacq
  - taylor
  - wanda


In [3]:
# Get a specific selector function
magnitude_selector = get_selector("magnitude")
random_selector = get_selector("random")

print(f"magnitude: {magnitude_selector}")
print(f"random:    {random_selector}")

magnitude: <function magnitude_selector at 0x7fe4a6e43d80>
random:    <function random_selector at 0x7fe4a6e43ba0>


### Registering a Custom Selector

You can register your own selector with `@register("name")`. A selector is a callable with signature `(model, task, config_dict) -> Dict[str, float]`.

In [4]:
from circuitkit.selection import register

@register("my_custom")
def my_custom_selector(model, task, config):
    """Example selector: (model, task, config_dict) -> Dict[str, float].

    In practice, use `model` and `task` to compute real importance scores
    (e.g. activation norms, gradients). This toy version returns uniform
    placeholder scores.
    """
    n_layers = getattr(getattr(model, "cfg", None), "n_layers", 10)
    return {f"node_{i}": 1.0 / (i + 1) for i in range(n_layers)}

# Verify registration
print(f"Now {len(list_selectors())} selectors registered")
assert "my_custom" in list_selectors()

Now 15 selectors registered


---
## Section B: MasterGrid

The `MasterGrid` runs a (methods × wrappers × seeds) experiment grid. It's designed for systematic comparison across the full CircuitKIT toolkit.

A full grid on a real model is expensive, so here we demonstrate the API with a small configuration.

In [5]:
from circuitkit.evaluation.master_grid import MasterGrid, MasterGridCell

In [6]:
# Small demo grid — 2 methods × 2 wrappers × 2 seeds
grid = MasterGrid(
    methods=["eap", "eap-ig"],
    model="gpt2",
    wrappers=["pruning", "quantization"],
    seeds=[42, 123],
)

In [7]:
# Run the grid (small scale — takes ~2-3 min on CPU)
import os
os.makedirs("./results/master_grid", exist_ok=True)

try:
    grid.run(out_dir="./results/master_grid")
    print(f"Grid completed: {len(grid.cells)} cells")
except Exception as e:
    print(f"Grid run failed (expected on limited hardware): {e}")
    print("Proceeding with synthetic data for demo...")

    # Populate with synthetic results for demonstration
    import random
    grid.cells = []
    for m in ["eap", "eap-ig"]:
        for w in ["pruning", "quantization"]:
            for s in [42, 123]:
                grid.cells.append(MasterGridCell(
                    method=m, wrapper=w, seed=s,
                    quality=random.uniform(0.5, 0.9),
                    stability=random.uniform(0.3, 0.8),
                    faithfulness=random.uniform(0.4, 0.85),
                ))

07:31:27 • MasterGrid.run(): 8 cells | methods=['eap', 'eap-ig'], wrappers=['pruning', 'quantization'], seeds=[42, 123]
07:31:27 •   running: eap x pruning x seed=42
07:31:27 • ==================================================
07:31:27 • Starting: Circuit Discovery
07:31:27 • ==================================================
07:31:27 • [1/4] Loading and validating configuration
07:31:27 ⚙ Configuration validated successfully.
07:31:27 ⚙ Config: EAP on ioi task, node level, 10% sparsity
07:31:27 • [2/4] Setting up model (model=gpt2)
Loaded pretrained model gpt2 into HookedTransformer
07:31:30 🔧 Model loading: 3.06s
07:31:30 🔧 Model: gpt2 (163.0M params, cuda device)
07:31:30 • [3/4] Running discovery algorithm (algorithm=eap)
07:31:30 • Starting EAP discovery algorithm
07:31:30 • Loading cached IOI data for gpt2 from cache/ioi/ioi_gpt2_32_seed42.csv


100%|██████████| 32/32 [00:03<00:00, 10.56it/s]

07:31:34 • Converted 156 node scores from EAP-IG graph to CircuitKIT format.
07:31:34 • Saved unified CircuitScores → circuit_discovery_results_scores.json
07:31:34 • Saved 'scores' → circuit_discovery_results_scores.pt
07:31:34 • [4/4] Identifying nodes to prune
07:31:34 • 
--- Running Categorized Node Pruning (Scope: HEADS) ---
07:31:34 • Found 144 Attention Heads and 12 MLP layers to consider for pruning.
07:31:34 • Skipping MLP layer pruning as per scope.
07:31:34 • Targeting to prune 14 out of 144 total Attention Heads (10%).
07:31:34 • Actually pruned 14 Attention Heads.
07:31:34 •   - Layer 1: Pruned 4/12 heads.
07:31:34 •   - Layer 2: Pruned 2/12 heads.
07:31:34 •   - Layer 3: Pruned 1/12 heads.
07:31:34 •   - Layer 4: Pruned 3/12 heads.
07:31:34 •   - Layer 5: Pruned 1/12 heads.
07:31:34 •   - Layer 7: Pruned 1/12 heads.
07:31:34 •   - Layer 8: Pruned 1/12 heads.
07:31:34 •   - Layer 9: Pruned 1/12 heads.
07:31:34 • --- Categorized Pruning Finished ---
07:31:34 •   Pruned 14 n

Loaded pretrained model gpt2 into HookedTransformer
07:31:37 🔧 Model loading: 2.95s
07:31:37 🔧 Model: gpt2 (163.0M params, cuda device)
07:31:37 • [3/4] Running discovery algorithm (algorithm=eap)
07:31:37 • Starting EAP discovery algorithm
07:31:37 • Generating new IOI data for gpt2 (samples=32, seed=123)
07:31:37 • Saved generated data to cache: cache/ioi/ioi_gpt2_32_seed123.csv


100%|██████████| 32/32 [00:02<00:00, 11.45it/s]

...

100%|██████████| 32/32 [00:09<00:00,  3.28it/s]

07:32:32 • Converted 156 node scores from EAP-IG graph to CircuitKIT format.
07:32:32 • Saved unified CircuitScores → circuit_discovery_results_scores.json
07:32:32 • Saved 'scores' → circuit_discovery_results_scores.pt
07:32:32 • [4/4] Identifying nodes to prune
07:32:32 • 
--- Running Categorized Node Pruning (Scope: HEADS) ---
07:32:32 • Found 144 Attention Heads and 12 MLP layers to consider for pruning.
07:32:32 • Skipping MLP layer pruning as per scope.
07:32:32 • Targeting to prune 14 out of 144 total Attention Heads (10%).
07:32:32 • Actually pruned 14 Attention Heads.
07:32:32 •   - Layer 0: Pruned 1/12 heads.
07:32:32 •   - Layer 1: Pruned 1/12 heads.
07:32:32 •   - Layer 2: Pruned 2/12 heads.
07:32:32 •   - Layer 3: Pruned 2/12 heads.
07:32:32 •   - Layer 4: Pruned 2/12 heads.
07:32:32 •   - Layer 5: Pruned 1/12 heads.
07:32:32 •   - Layer 7: Pruned 3/12 heads.
07:32:32 •   - Layer 8: Pruned 2/12 heads.
07:32:32 • --- Categorized Pruning Finished ---
07:32:32 •   Pruned 14 n

Loaded pretrained model gpt2 into HookedTransformer
07:32:35 🔧 Model loading: 2.84s
07:32:35 🔧 Model: gpt2 (163.0M params, cuda device)
07:32:35 • [3/4] Running discovery algorithm (algorithm=eap-ig)
07:32:35 • Starting EAP-IG discovery algorithm
07:32:35 • Loading cached IOI data for gpt2 from cache/ioi/ioi_gpt2_32_seed123.csv


100%|██████████| 32/32 [00:09<00:00,  3.21it/s]

07:32:45 • Converted 156 node scores from EAP-IG graph to CircuitKIT format.
07:32:45 • Saved unified CircuitScores → circuit_discovery_results_scores.json
07:32:45 • Saved 'scores' → circuit_discovery_results_scores.pt
07:32:45 • [4/4] Identifying nodes to prune
07:32:45 • 
--- Running Categorized Node Pruning (Scope: HEADS) ---
07:32:45 • Found 144 Attention Heads and 12 MLP layers to consider for pruning.
07:32:45 • Skipping MLP layer pruning as per scope.
07:32:45 • Targeting to prune 14 out of 144 total Attention Heads (10%).
07:32:45 • Actually pruned 14 Attention Heads.
07:32:45 •   - Layer 0: Pruned 1/12 heads.
07:32:45 •   - Layer 1: Pruned 2/12 heads.
07:32:45 •   - Layer 4: Pruned 3/12 heads.
07:32:45 •   - Layer 5: Pruned 1/12 heads.
07:32:45 •   - Layer 7: Pruned 2/12 heads.
07:32:45 •   - Layer 8: Pruned 3/12 heads.
07:32:45 •   - Layer 9: Pruned 1/12 heads.
07:32:45 •   - Layer 11: Pruned 1/12 heads.
07:32:45 • --- Categorized Pruning Finished ---
07:32:45 •   Pruned 14 

In [8]:
# View as DataFrame
df = grid.to_dataframe()
df

,method,wrapper,seed,quality,stability,faithfulness,baseline_ratio,probe
0,eap,pruning,42,None,None,None,None,None
1,eap,pruning,123,None,None,None,None,None
2,eap,quantization,42,None,None,None,None,None
3,eap,quantization,123,None,None,None,None,None
4,eap-ig,pruning,42,None,None,None,None,None
5,eap-ig,pruning,123,None,None,None,None,None
6,eap-ig,quantization,42,None,None,None,None,None
7,eap-ig,quantization,123,None,None,None,None,None


In [9]:
# Per-wrapper ranking summary
print(grid.summary())

MasterGrid(2 methods x 2 wrappers, model=gpt2)
         pruning: winner = ?
    quantization: winner = ?


In [10]:
# Per-wrapper ranks dict
ranks = grid.per_wrapper_ranks()
for (method, wrapper), rank in sorted(ranks.items()):
    print(f"  {method:>8s} × {wrapper:>14s} → rank {rank}")

### Loading Pre-Computed Results

For large grids, results are typically computed via shell scripts and aggregated later.

`MasterGrid.from_csv("path/to/aggregated_grid.csv")` loads a CSV with columns: `method`, `wrapper`, `seed`, `quality`, `stability`, ...

---
## Section C: Intervention Faithfulness (IF) Metric

The IF metric predicts whether a circuit will produce good downstream interventions, using three cheap signals:

$$IF(c, w) = \alpha \cdot J_5(c) + \beta \cdot R_3(c) + \gamma \cdot P(c, w)$$

where $J_5$ is Pillar-5 stability, $R_3$ is Pillar-3 baseline ratio, and $P$ is a low-budget intervention probe.

Coefficients are fit by leave-one-wrapper-out cross-validation.

In [11]:
from circuitkit.evaluation.intervention_faithfulness import IF, IFResult

In [12]:
# Construct a synthetic cells dict for demonstration
# Format: {(method, wrapper): {"stability": float, "baseline_ratio": float,
#           "probe": float, "quality": float, "faithfulness": float}}
import random
random.seed(42)

methods = ["eap", "eap-ig", "eap-gp"]
wrappers = ["pruning", "quantization", "lora", "steering"]

cells = {}
for m in methods:
    for w in wrappers:
        cells[(m, w)] = {
            "stability": random.uniform(0.3, 0.9),
            "baseline_ratio": random.uniform(0.5, 1.5),
            "probe": random.uniform(0.2, 0.8),
            "quality": random.uniform(0.4, 0.9),
            "faithfulness": random.uniform(0.4, 0.9),
        }

print(f"Cells: {len(cells)} entries ({len(methods)} methods × {len(wrappers)} wrappers)")

Cells: 12 entries (3 methods × 4 wrappers)


In [13]:
# Fit the IF metric with leave-one-wrapper-out CV
if_metric = IF(simplex_grid_step=0.1)
result = if_metric.fit_loo(cells)

print(f"IF accuracy:          {result.if_accuracy:.1%}")
print(f"Faith-only accuracy:  {result.faith_accuracy:.1%}")
print(f"Stab-only accuracy:   {result.stab_accuracy:.1%}")
print(f"Chance (top-1):       {result.chance_top1:.1%}")

IF accuracy:          25.0%
Faith-only accuracy:  50.0%
Stab-only accuracy:   25.0%
Chance (top-1):       33.3%


In [14]:
# Per-fold details — compare IF prediction against faithfulness-only and stability-only baselines
header = f"{'Held wrapper':<14} {'Coeffs (a,b,g)':<20} {'True':>10} {'IF':>10} {'Faith':>10} {'Stab':>10}"
print(header)
print("-" * len(header))
for fold in result.folds:
    coeffs_str = f"({fold.coeffs[0]:.2f},{fold.coeffs[1]:.2f},{fold.coeffs[2]:.2f})"
    print(f"{fold.held_wrapper:<14} {coeffs_str:<20} {fold.true_winner:>10} "
          f"{fold.pred_if:>10} {fold.pred_faithfulness_only:>10} {fold.pred_stability_only:>10}")

Held wrapper   Coeffs (a,b,g)             True         IF      Faith       Stab
-------------------------------------------------------------------------------
lora           (0.00,0.10,0.90)         eap-ig     eap-ig        eap     eap-ig
pruning        (0.00,0.10,0.90)         eap-gp     eap-ig     eap-ig     eap-ig
quantization   (0.00,0.10,0.90)         eap-ig     eap-gp     eap-ig        eap
steering       (0.00,0.00,1.00)         eap-ig     eap-gp     eap-ig     eap-gp


---
## Section D: Pipeline Reuse — Iterate Without Re-Discovering

Once you've discovered a circuit, you can reload it and try different interventions without running discovery again.

In [15]:
# First, create an artifact to work with
pipe_orig = Pipeline("gpt2", task="ioi", precision="float32", output_dir="./results/reuse")
pipe_orig.discover(algorithm="eap-ig", n_examples=32, batch_size=4, ig_steps=5)

artifact_path = pipe_orig._artifact_path
print(f"Artifact saved at: {artifact_path}")

Loaded pretrained model gpt2 into HookedTransformer
07:32:48 • ==================================================
07:32:48 • Starting: Circuit Discovery
07:32:48 • ==================================================
07:32:48 • [1/4] Loading and validating configuration
07:32:48 ⚙ Configuration validated successfully.
07:32:48 ⚙ Config: EAP-IG on ioi task, node level, 30% sparsity
07:32:48 • [2/4] Setting up model (model=gpt2)
Loaded pretrained model gpt2 into HookedTransformer
07:32:51 🔧 Model loading: 3.00s
07:32:51 🔧 Model: gpt2 (163.0M params, cuda device)
07:32:51 • [3/4] Running discovery algorithm (algorithm=eap-ig)
07:32:51 • Starting EAP-IG discovery algorithm
07:32:51 • Loading cached IOI data for gpt2 from cache/ioi/ioi_gpt2_32_seed42.csv


100%|██████████| 8/8 [00:02<00:00,  3.39it/s]

07:32:54 • Converted 156 node scores from EAP-IG graph to CircuitKIT format.
07:32:54 • Saved unified CircuitScores → results/reuse/eap-ig_gpt2_ioi_node_scores.json
07:32:54 • Saved 'scores' → results/reuse/eap-ig_gpt2_ioi_node_scores.pt
07:32:54 • [4/4] Identifying nodes to prune
07:32:54 • 
--- Running Categorized Node Pruning (Scope: BOTH) ---
07:32:54 • Found 144 Attention Heads and 12 MLP layers to consider for pruning.
07:32:54 • Pruning 3 out of 12 MLP layers (30%).
07:32:54 • Targeting to prune 43 out of 144 total Attention Heads (30%).
07:32:54 • Actually pruned 43 Attention Heads.
07:32:54 •   - Layer 0: Pruned 2/12 heads.
07:32:54 •   - Layer 1: Pruned 6/12 heads.
07:32:54 •   - Layer 2: Pruned 5/12 heads.
07:32:54 •   - Layer 3: Pruned 6/12 heads.
07:32:54 •   - Layer 4: Pruned 6/12 heads.
07:32:54 •   - Layer 5: Pruned 2/12 heads.
07:32:54 •   - Layer 6: Pruned 2/12 heads.
07:32:54 •   - Layer 7: Pruned 3/12 heads.
07:32:54 •   - Layer 8: Pruned 4/12 heads.
07:32:54 •   - 

### `Pipeline.from_artifact()` — Reload and Apply

In [16]:
# Reload the circuit and prune at different sparsity
pipe_v2 = Pipeline.from_artifact(
    artifact_path,
    model_name="gpt2",
    task="ioi",
    precision="float32",
    output_dir="./results/reuse/v2",
)

print(f"Loaded circuit: {pipe_v2._circuit}")

# Apply a different sparsity without re-discovering
pipe_v2.prune(sparsity=0.5, scope="both")
pipe_v2.export("./results/reuse/v2/checkpoint_sparse50")

pipe_v2.summary()

Loaded circuit: Circuit(level=node, n_nodes=46, algorithm='eap-ig', task='ioi', model='gpt2', n_scored=156)
Loaded pretrained model gpt2 into HookedTransformer
07:32:57 • Starting structural masking: eap-ig scores on gpt2 (inplace=False)
07:32:57 • Masking 78 nodes: ['A3.10', 'A4.9', 'A3.11', 'A5.1', 'A0.7'] + 73 more
07:32:57 ✓ Masking complete. Effective sparsity: 9.41%


                 Pipeline Summary                 
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Property      ┃ Value                          ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Model         │ gpt2                           │
│ Task          │ ioi                            │
│ Precision     │ float32                        │
│ Output Dir    │ ./results/reuse/v2             │
│ Steps         │ from_artifact → prune → export │
│ Circuit Level │ node                           │
│ Circuit Size  │ 46                             │
│ Algorithm     │ eap-ig                         │
└───────────────┴────────────────────────────────┘

### `Pipeline.from_scores()` — Load from Scores File

In [17]:
# The scores side-car (_scores.pt) can also be loaded
from pathlib import Path

scores_pt = Path(artifact_path).parent / f"{Path(artifact_path).stem}_scores.pt"
if scores_pt.exists():
    pipe_v3 = Pipeline.from_scores(
        scores_pt,
        model_name="gpt2",
        task="ioi",
        precision="float32",
    )
    print(f"Loaded from scores: {pipe_v3._circuit}")
else:
    print(f"Scores file not found at {scores_pt}")
    print("This happens when discovery doesn't produce a _scores.pt side-car.")

Loaded from scores: Circuit(level=node, n_nodes=0, model='gpt2', n_scored=0)


---
## Summary

| Tool | Purpose | When to use |
|------|---------|-------------|
| Selector Registry | Access / register node-ranking functions | Custom baselines, new algorithms |
| MasterGrid | Systematic (method × wrapper × seed) comparison | Paper experiments |
| IF Metric | Predict intervention quality from cheap signals | Method selection |
| `from_artifact` / `from_scores` | Reload without re-discovering | Iteration, sparsity sweeps |